# Challenge:

- Build a synthetic dataset generator, write models that can generate datasets.
- Use a variety of models and prompts for diverse output, try quantizing and not quantizing, try different shapes and sizes.
- Create a Gradio UI for your product

## Idea

- Generate sample data for use when developing a REST API.

### Input: API spec, in different formats.

(API spec is already known vs suggest structure and endpoints based on model?)
  
- Empty JSON structure
- Java
- OAD (OpenAPI document) - in JSON or YAML
- UML? (Image or Mermaid chart def)
- URL - for example running Swagger UI instance
- As GUI - dropdowns or other UI elements to select entities and relations

In [ ]:
!uv pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6 huggingface_hub[hf_xet]

In [2]:
# Imports

from transformers import (
    pipeline,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
import os
from dotenv import load_dotenv
from openai import OpenAI
from huggingface_hub import login

load_dotenv(override=True)

True

In [ ]:
# Constants

LLAMA_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
QWEN_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
PHI_MODEL = "microsoft/Phi-4-mini-instruct"
OPENAPI_FILE = "storefront-sample.json"

In [4]:
# Huggingface login

login(token=os.getenv("HF_TOKEN"), add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [5]:
# in case $ref is used in the schema - resolve_ref


def resolve_ref(spec, ref):
    path = ref.replace("#/", "").split("/")
    obj = spec
    for p in path:
        obj = obj[p]
    return obj

In [6]:
# Extraction function

import json


def extract_schema(openapi, path, method):
    method = method.lower()
    endpoint = openapi["paths"][path][method]
    schema = endpoint["requestBody"]["content"]["application/json"]["schema"]
    return schema


with open(OPENAPI_FILE) as f:
    spec = json.load(f)

schema = extract_schema(spec, "/cart/items", "post")

if "$ref" in schema:
    schema = resolve_ref(spec, schema["$ref"])

print(json.dumps(schema, indent=2))

{
  "type": "object",
  "required": [
    "product_id",
    "quantity"
  ],
  "properties": {
    "product_id": {
      "type": "string",
      "format": "uuid"
    },
    "quantity": {
      "type": "integer",
      "minimum": 1
    }
  }
}


In [7]:
user_prompt = f"""Generate realistic JSON objects following this schema:
{schema}
"""
system_prompt = """
You excel at generating realistic JSON objects following a given schema.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [8]:
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)


def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name, device_map="auto", quantization_config=quant_config
    )
    return model, tokenizer


def generate_json(model, tokenizer, messages, max_new_tokens=500):
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True
    ).to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
import re


def extract_json(text):
    """Pull the first complete JSON object or array out of model output."""
    if text is None:
        return None

    # Try ```json ... ``` code blocks first
    code_block = re.search(r"```(?:json)?\s*(\{[\s\S]*?\}|\[[\s\S]*?\])\s*```", text)
    if code_block:
        return code_block.group(1).strip()

    # Fall back to matching outermost braces/brackets
    for start, ch in enumerate(text):
        if ch not in "{[":
            continue
        close = "}" if ch == "{" else "]"
        depth = 0
        for end in range(start, len(text)):
            if text[end] == ch:
                depth += 1
            elif text[end] == close:
                depth -= 1
                if depth == 0:
                    return text[start : end + 1]
    return None


def validate_json(text):
    """Parse JSON string, return (parsed_object, is_valid) tuple."""
    try:
        return json.loads(text), True
    except (json.JSONDecodeError, TypeError):
        return None, False

In [ ]:
import gc

MODELS = {
    "Llama 3.1 8B": LLAMA_MODEL,
    "Qwen 2.5 Coder 7B": QWEN_MODEL,
    "Phi 2.5 Coder": PHI_MODEL,
}

results = []

for label, model_name in MODELS.items():
    print(f"\n{'=' * 60}")
    print(f"  {label} ({model_name})")
    print(f"{'=' * 60}")

    try:
        model, tokenizer = load_model(model_name)
        raw_output = generate_json(model, tokenizer, messages)

        extracted = extract_json(raw_output)
        parsed, is_valid = validate_json(extracted)

        results.append(
            {
                "model": label,
                "valid_json": is_valid,
                "raw_output": raw_output,
                "extracted": extracted,
                "parsed": parsed,
            }
        )

        print(f"\nRaw output:\n{raw_output}")
        print(f"\nExtracted JSON:\n{extracted}")
        print(f"\nValid JSON: {is_valid}")

        del model, tokenizer
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"\nError: {e}")
        results.append(
            {
                "model": label,
                "valid_json": False,
                "raw_output": None,
                "extracted": None,
                "parsed": None,
                "error": str(e),
            }
        )

In [ ]:
print(f"\n{'=' * 60}")
print("  Model Comparison Summary")
print(f"{'=' * 60}\n")
print(f"  {'Model':<25} {'Valid JSON':<14} {'Extracted'}")
print(f"  {'-' * 53}")
for r in results:
    has_output = r["extracted"] is not None
    print(f"  {r['model']:<25} {str(r['valid_json']):<14} {has_output}")